# DeepSceneLoc — Kaggle Training (T4 x2)

Trains all 3 models (EfficientNet-B0, ViT-B/16, ResNet-50) with full fine-tune + LLRD + SWA, then builds a calibrated ensemble.

**Before running:**
1. Notebook settings → **Accelerator = GPU T4 x2**, **Internet = ON**.
2. **Add Input** → Add dataset: https://www.kaggle.com/datasets/nickj26/places2-mit-dataset
3. Make sure the `dev-krishan` branch is pushed to GitHub (this notebook clones it).

Scripts auto-detect 2 GPUs → `nn.DataParallel`. `--batch` is the **total** batch across both GPUs.

> Kaggle sessions cap at ~9–12 h. A 45-epoch full fine-tune is heavy — prefer **one model per session** and use `--auto-resume` to continue.

## Clone code + install deps

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/DeepSceneLoc/DeepSceneLoc-Visual-Scene-Understanding-for-Image-Based-Location-Estimation-Using-Deep-Learning.git"
BRANCH   = "dev-krishan"

import os
if not os.path.isdir("repo"):
    !git clone -b $BRANCH $REPO_URL repo
%cd repo

# torch/torchvision are preinstalled on the Kaggle GPU image; add the rest.
!pip install -q "numpy<2" timm scikit-learn seaborn plotly google-generativeai

## Cell 2 — Sanity check: GPUs + repo layout

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))

!ls scripts/training/
print("--- available Kaggle inputs ---")
!ls /kaggle/input

## Cell 3 — Prepare data

**Dataset:** Places2 MIT Dataset (https://www.kaggle.com/datasets/nickj26/places2-mit-dataset)

Set `DATASET_DIR` to the dataset path. The Places2 dataset has 365 categories, but we only need 5:
- Coastal (beach, harbor, coast, etc.)
- Forest (forest, jungle, woodland, etc.)
- Mountain (mountain, peak, valley, etc.)
- Rural (farm, countryside, village, etc.)
- Urban (city, street, building, etc.)

Two cases:
- **Already filtered to 5 categories** with split into `train/ val/ test/<Category>/` → point `DATA` directly at it.
- **Full Places2 dataset** → You'll need to filter and map categories first (use your data preparation script).

Expected final layout: `data/processed/{train,val,test}/{Coastal,Forest,Mountain,Rural,Urban}/`

In [ ]:
# For Places2 MIT dataset:
DATASET_DIR = "/kaggle/input/places-2-mit-dataset"

# If you have a pre-filtered subset with only 5 categories:
# DATASET_DIR = "/kaggle/input/places2-mit-dataset/filtered_5_categories"

import os, pathlib

def _has_splits(p):
    return all(os.path.isdir(os.path.join(p, s)) for s in ("train", "val", "test"))

if _has_splits(DATASET_DIR):
    DATA = DATASET_DIR
    print("Dataset already split. Using DATA =", DATA)
else:
    DATA = "data/processed"
    print("Splitting organised folders -> data/processed ...")
    !python scripts/split_dataset.py --data "$DATASET_DIR" --out data/processed

# quick counts
for split in ("train", "val", "test"):
    root = pathlib.Path(DATA) / split
    if root.exists():
        n = sum(1 for _ in root.rglob("*.jpg")) + sum(1 for _ in root.rglob("*.png"))
        print(f"  {split}: {n} images")
print("DATA =", DATA)

## Smoke test FIRST (≈1 min)

Catches data/path/config bugs before you burn a 9-hour run. Runs 2 epochs × 5 batches.

In [ ]:
!python scripts/training/run_training_efficientnet_b0.py --dry-run --batch 8 --workers 2 --data "$DATA"

## Train EfficientNet-B0 (full fine-tune + LLRD + SWA)

`--batch 128` = 64/GPU on T4 x2. Add `--auto-resume` if continuing a previous session.

In [ ]:
!python scripts/training/run_training_efficientnet_b0.py \
    --data "$DATA" --batch 128 --epochs 45 --workers 4 \
    --full-finetune --swa

## Train ViT-B/16 (full fine-tune + LLRD + SWA)

ViT uses more VRAM → `--batch 96` (48/GPU). If you hit OOM, drop to `--batch 64`.

In [ ]:
!python scripts/training/run_training_vit_b16.py \
    --data "$DATA" --batch 96 --epochs 45 --workers 4 \
    --full-finetune --swa

## Train ResNet-50 baseline (discriminative LR)

In [ ]:
!python scripts/training/run_training_resnet50.py \
    --data "$DATA" --batch 128 --epochs 40 --workers 4 \
    --full-finetune

## Calibrated ensemble evaluation

Averages all 3 models with TTA + temperature calibration, weighted by val accuracy.
Uses `*_best.pth`; swap to `*_swa.pth` if the SWA checkpoint scored higher.

In [ ]:
!python scripts/training/run_ensemble_eval.py \
    --ckpts models/checkpoints/efficientnet/EfficientNet-B0_best.pth \
            models/checkpoints/vit/ViT-B_16_best.pth \
            models/checkpoints/resnet/best_model.pth \
    --data "$DATA" --weight-by-val-acc --calibrate

## Cell 9 — Collect artifacts (→ notebook Output)

Zips checkpoints, results, and logs into `/kaggle/working/`. Download them from the **Output** tab after the run, or via `kaggle kernels output <user>/<kernel>`.

In [ ]:
import shutil, os, glob

for name in ["models", "results", "logs"]:
    if os.path.isdir(name):
        out = shutil.make_archive(f"/kaggle/working/{name}", "zip", ".", name)
        print("zipped ->", out, f"({os.path.getsize(out)/1e6:.1f} MB)")
    else:
        print("skip (missing):", name)

print("\nKey files to download:")
for pat in ["models/checkpoints/**/*.pth",
            "results/**/metrics/*.json",
            "results/ensemble_*/ensemble_evaluation.json",
            "logs/**/*_history.json"]:
    for f in glob.glob(pat, recursive=True):
        print("  ", f)

## Resume across sessions (9 h limit)

If a run gets cut off, start a new session, re-run Cells 1–3, then re-run the training cell **with** `--auto-resume` — it finds the latest checkpoint for that model and continues:

```bash
!python scripts/training/run_training_efficientnet_b0.py --data "$DATA" --batch 128 --epochs 45 --workers 4 --full-finetune --swa --auto-resume
```

Checkpoints must survive between sessions: either **commit** the notebook (saves `/kaggle/working`) or persist `models/` to a Kaggle Dataset you re-attach next session.

### Collecting back to your local repo
Download `models.zip`, unzip so files land at `models/checkpoints/{efficientnet,vit,resnet}/*.pth` in your local clone. The demo apps and local `run_ensemble_eval.py` then work offline.